## Obtaining, parsing and understanding ALT loci

ALT loci are included in the GRCh37 and GRCh38 assemblies.  These represent alternative assemblies of different portions of the human reference genome.  They are especially concentrated in hypervariable regions of the genome, like MHC and KIR.  The ALT loci -- their sequences and alignments to the reference -- are available via FTP.  This notebook investigates how to download the ALT loci along with their alignment to the reference and the reference sequence itself.  It also investigates the formats of the various files involved and how to parse them.  Finally, it shows how to convert the CIGAR-style alignments in the ALT-locus GFF files into a stacked alignment more appropriate for visualization.

In [1]:
import os
import gzip
import tempfile
import logging
import collections
import urllib.request
from collections import defaultdict
from ftplib import FTP

In [2]:
logging.basicConfig(level=logging.DEBUG)

In [3]:
def nav_to_root():
    ftp = FTP('ftp.ncbi.nlm.nih.gov')
    ftp.login()
    dr = 'genomes/all/GCA/000/001/405/GCA_000001405.28_GRCh38.p13/GCA_000001405.28_GRCh38.p13_assembly_structure'
    ftp.cwd(dr)
    return ftp

In [4]:
alt_grps = []

def handle_line(st):
    fn = st.split()[-1]
    if fn.startswith('ALT_'):
        alt_grps.append(fn)

nav_to_root().retrlines('LIST', handle_line)

'226 Transfer complete'

Names like `ALT_REF_LOCI_1` are "alternate locus groups"

In [5]:
alt_grps

['ALT_REF_LOCI_1',
 'ALT_REF_LOCI_10',
 'ALT_REF_LOCI_11',
 'ALT_REF_LOCI_12',
 'ALT_REF_LOCI_13',
 'ALT_REF_LOCI_14',
 'ALT_REF_LOCI_15',
 'ALT_REF_LOCI_16',
 'ALT_REF_LOCI_17',
 'ALT_REF_LOCI_18',
 'ALT_REF_LOCI_19',
 'ALT_REF_LOCI_2',
 'ALT_REF_LOCI_20',
 'ALT_REF_LOCI_21',
 'ALT_REF_LOCI_22',
 'ALT_REF_LOCI_23',
 'ALT_REF_LOCI_24',
 'ALT_REF_LOCI_25',
 'ALT_REF_LOCI_26',
 'ALT_REF_LOCI_27',
 'ALT_REF_LOCI_28',
 'ALT_REF_LOCI_29',
 'ALT_REF_LOCI_3',
 'ALT_REF_LOCI_30',
 'ALT_REF_LOCI_31',
 'ALT_REF_LOCI_32',
 'ALT_REF_LOCI_33',
 'ALT_REF_LOCI_34',
 'ALT_REF_LOCI_35',
 'ALT_REF_LOCI_4',
 'ALT_REF_LOCI_5',
 'ALT_REF_LOCI_6',
 'ALT_REF_LOCI_7',
 'ALT_REF_LOCI_8',
 'ALT_REF_LOCI_9']

### Get alignment GFFs

In [6]:
gffs, grp_gffs = [], []
ftp = nav_to_root()
for alt_grp in alt_grps:
    dr_alt = os.path.join(alt_grp, 'alt_scaffolds', 'alignments')
    ftp.cwd(dr_alt)

    def handle_alignment_line(st):
        fn = st.split()[-1]
        if fn.endswith('.gff'):
            grp_gffs.append((alt_grp, fn))
            gffs.append(fn)

    ftp.retrlines('LIST', handle_alignment_line)
    ftp.cwd('../../..')

In [7]:
print(len(gffs), len(set(gffs)))

261 261


Is this the expected number of ALT contigs?

In [8]:
gffs[0]

'GL000250.2_CM000668.2.gff'

In [9]:
srcs, dsts = [], []
for gff in gffs:
    toks = gff[:-4].split('_')
    srcs.append(toks[0])
    dsts.append(toks[1])

In [10]:
print(len(srcs), len(set(srcs)))

261 261


In [11]:
print(len(dsts), len(set(dsts)))

261 23


In [12]:
dst_counter = collections.Counter(dsts)

In [13]:
for k, v in dst_counter.items():
    print((k, v))

('CM000668.2', 16)
('CM000666.2', 11)
('CM000679.2', 18)
('CM000667.2', 13)
('CM000663.2', 12)
('CM000664.2', 15)
('CM000665.2', 16)
('CM000669.2', 9)
('CM000671.2', 5)
('CM000672.2', 4)
('CM000673.2', 12)
('CM000674.2', 13)
('CM000677.2', 9)
('CM000678.2', 6)
('CM000680.2', 10)
('CM000681.2', 43)
('CM000682.2', 4)
('CM000683.2', 7)
('CM000684.2', 9)
('CM000670.2', 16)
('CM000675.2', 6)
('CM000676.2', 4)
('CM000685.2', 3)


In [14]:
dst_counter.most_common(1)[0][0]

'CM000681.2'

In [15]:
# Parse attributes field of GFF line
def parse_attr(attr):
    targstr = ';Target='
    assert targstr in attr, attr
    toff = attr.index(targstr)
    target = attr[toff+len(targstr):attr.index(' ', toff)]
    gapstr = ';Gap='
    assert gapstr in attr, attr
    coff = attr.index(gapstr)
    cigar_st = attr[coff+len(gapstr):]
    cigar_ls = cigar_st.split(' ')
    ops = list(map(lambda x: x[0], cigar_ls))
    for op in ops:
        assert op in 'MDI'
    runs = list(map(lambda x: int(x[1:]), cigar_ls))
    return target, list(zip(ops, runs))

In [16]:
# See if we can parse this example GFF line
attr='ID=aln0;Target=GL000250.2 1 4672374 +;align_id=1411;batch_id=142943;bit_score=1.12029;common_component=0;e_value=4.83307e+11;expansion=1.96768;filter_score=7;merge_aligner=1;merge_options=58;num_ident=2348261;num_mismatch=6441;pct_coverage=50.3963;pct_identity_gap=33.7832;pct_identity_gapopen_only=99.688;pct_identity_ungap=99.7265;reciprocity=3;Gap=M44558 D8 M13552 D2 M3960 D4 M2109 D2 M33 I2 M25649 I6 M18550 D1 M68504 I1 M25289 I44253 D44253 M3450 D1 M345 I1 M3095 I39 M16670 I1 M48609 D2 M2804 I29384 D29384 M14754 I7 M3825 D2 M7033 D8 M3588 D1 M19578 D1 M1324 D2 M5435 I5 M2064 I2 M5713 D1 M3146 I1 M15 I1 M28 I1 M2026 I1 M14932 I1 M168 D1 M6579 D4 M683 I8 M47 I4 M1147 I87598 D87598 M5106 I2 M1514 D1 M3116 D1 M5039 D1 M8561 I19 M8035 I1 M2049 I1 M502 I2 M3290 I10 M6685 I1 M8656 I104192 D104192 M1686 D2 M5376 D1 M19253 I1 M7864 I1 M41 D6 M4451 I1 M27910 I1 M9946 D1 M17202 I6 M1218 I1 M3967 D2 M462 I1 M20759 I1 M3779 I1 M1268 I1 M1551 I2 M4147 I22 M1757 I1 M3444 I1 M6666 D4 M1349 D4 M1101'

In [17]:
parse_attr(attr)

('GL000250.2',
 [('M', 44558),
  ('D', 8),
  ('M', 13552),
  ('D', 2),
  ('M', 3960),
  ('D', 4),
  ('M', 2109),
  ('D', 2),
  ('M', 33),
  ('I', 2),
  ('M', 25649),
  ('I', 6),
  ('M', 18550),
  ('D', 1),
  ('M', 68504),
  ('I', 1),
  ('M', 25289),
  ('I', 44253),
  ('D', 44253),
  ('M', 3450),
  ('D', 1),
  ('M', 345),
  ('I', 1),
  ('M', 3095),
  ('I', 39),
  ('M', 16670),
  ('I', 1),
  ('M', 48609),
  ('D', 2),
  ('M', 2804),
  ('I', 29384),
  ('D', 29384),
  ('M', 14754),
  ('I', 7),
  ('M', 3825),
  ('D', 2),
  ('M', 7033),
  ('D', 8),
  ('M', 3588),
  ('D', 1),
  ('M', 19578),
  ('D', 1),
  ('M', 1324),
  ('D', 2),
  ('M', 5435),
  ('I', 5),
  ('M', 2064),
  ('I', 2),
  ('M', 5713),
  ('D', 1),
  ('M', 3146),
  ('I', 1),
  ('M', 15),
  ('I', 1),
  ('M', 28),
  ('I', 1),
  ('M', 2026),
  ('I', 1),
  ('M', 14932),
  ('I', 1),
  ('M', 168),
  ('D', 1),
  ('M', 6579),
  ('D', 4),
  ('M', 683),
  ('I', 8),
  ('M', 47),
  ('I', 4),
  ('M', 1147),
  ('I', 87598),
  ('D', 87598),
  ('M'

In [18]:
class Alignment(object):
    def __init__(self, seqname, target, st, en, cigar):
        self.seqname = seqname
        self.target = target
        self.st = int(st) - 1
        self.en = int(en)
        assert self.st < self.en
        self.cigar = cigar
    
    def __repr__(self):
        return '%s[%d:%d]->%s' % (self.seqname, self.st, self.en, self.target)

In [19]:
# Now let's actually parse the GFF alignments
tmp = tempfile.mkdtemp()
ftp = nav_to_root()
als = defaultdict(list)
for alt_grp, gff in grp_gffs:
    tmpfn = os.path.join(tmp, gff)
    ftp.cwd(os.path.join(alt_grp, 'alt_scaffolds', 'alignments'))
    with open(tmpfn, 'wb') as tmpfh:
        logging.info('Retrieving GFF "%s"...' % gff)
        ftp.retrbinary('RETR ' + gff, tmpfh.write, 1024)
    ftp.cwd('../../..')
    with open(tmpfn, 'rt') as fh:
        for ln in fh:
            if ln[0] == '#':
                continue
            toks = ln.rstrip().split('\t')
            assert 9 == len(toks), toks
            seqname, source, feature, st_str, en_str, score, strand, frame, attr = toks
            assert 'Genbank' == source
            assert 'match' == feature
            # Score is sometimes non-0; not sure what it's meant to convey
            #assert '0' == score
            assert '+' == strand
            assert '.' == frame
            if 'pct_identity_gap=100' in attr:
                logging.warning('SKIPPING alignment due to 100% identity')
                continue
            target, cigar = parse_attr(attr)
            als[seqname].append(Alignment(seqname, target, st_str, en_str, cigar))

INFO:root:Retrieving GFF "GL000250.2_CM000668.2.gff"...
INFO:root:Retrieving GFF "GL000257.2_CM000666.2.gff"...
INFO:root:Retrieving GFF "GL000258.2_CM000679.2.gff"...
INFO:root:Retrieving GFF "GL339449.2_CM000667.2.gff"...
INFO:root:Retrieving GFF "GL383518.1_CM000663.2.gff"...
INFO:root:Retrieving GFF "GL383519.1_CM000663.2.gff"...
INFO:root:Retrieving GFF "GL383520.2_CM000663.2.gff"...
INFO:root:Retrieving GFF "GL383521.1_CM000664.2.gff"...
INFO:root:Retrieving GFF "GL383522.1_CM000664.2.gff"...
INFO:root:Retrieving GFF "GL383526.1_CM000665.2.gff"...
INFO:root:Retrieving GFF "GL383527.1_CM000666.2.gff"...
INFO:root:Retrieving GFF "GL383528.1_CM000666.2.gff"...
INFO:root:Retrieving GFF "GL383530.1_CM000667.2.gff"...
INFO:root:Retrieving GFF "GL383531.1_CM000667.2.gff"...
INFO:root:Retrieving GFF "GL383532.1_CM000667.2.gff"...
INFO:root:Retrieving GFF "GL383533.1_CM000668.2.gff"...
INFO:root:Retrieving GFF "GL383534.2_CM000669.2.gff"...
INFO:root:Retrieving GFF "GL383539.1_CM000671.2.

INFO:root:Retrieving GFF "KI270840.1_CM000675.2.gff"...
INFO:root:Retrieving GFF "KI270841.1_CM000675.2.gff"...
INFO:root:Retrieving GFF "KI270842.1_CM000675.2.gff"...
INFO:root:Retrieving GFF "KI270843.1_CM000675.2.gff"...
INFO:root:Retrieving GFF "KI270844.1_CM000676.2.gff"...
INFO:root:Retrieving GFF "KI270845.1_CM000676.2.gff"...
INFO:root:Retrieving GFF "KI270846.1_CM000676.2.gff"...
INFO:root:Retrieving GFF "KI270847.1_CM000676.2.gff"...
INFO:root:Retrieving GFF "KI270848.1_CM000677.2.gff"...
INFO:root:Retrieving GFF "KI270849.1_CM000677.2.gff"...
INFO:root:Retrieving GFF "KI270850.1_CM000677.2.gff"...
INFO:root:Retrieving GFF "KI270851.1_CM000677.2.gff"...
INFO:root:Retrieving GFF "KI270852.1_CM000677.2.gff"...
INFO:root:Retrieving GFF "KI270853.1_CM000678.2.gff"...
INFO:root:Retrieving GFF "KI270854.1_CM000678.2.gff"...
INFO:root:Retrieving GFF "KI270855.1_CM000678.2.gff"...
INFO:root:Retrieving GFF "KI270856.1_CM000678.2.gff"...
INFO:root:Retrieving GFF "KI270857.1_CM000679.2.

In [20]:
als

defaultdict(list,
            {'CM000668.2': [CM000668.2[28734407:33367716]->GL000250.2,
              CM000668.2[79350007:79446911]->GL383533.1,
              CM000668.2[168796075:168951646]->KB021644.2,
              CM000668.2[169724798:169919706]->KI270797.1,
              CM000668.2[170272762:170535221]->KI270798.1,
              CM000668.2[68262878:68402208]->KI270799.1,
              CM000668.2[65774813:65944033]->KI270800.1,
              CM000668.2[127654802:128521146]->KI270801.1,
              CM000668.2[166220579:166285980]->KI270802.1,
              CM000668.2[28510119:33383765]->GL000251.2,
              CM000668.2[28734407:33361299]->GL000252.2,
              CM000668.2[28734407:33258200]->GL000253.2,
              CM000668.2[28734407:33391865]->GL000254.2,
              CM000668.2[28734407:33411973]->GL000255.2,
              CM000668.2[28691465:33480577]->GL000256.2,
              CM000668.2[32385053:32454732]->KI270758.1],
             'CM000666.2': [CM000666.2[683063

In [21]:
def sizes_from_cigar(cigar):
    ilen, totlen, dlen = 0, 0, 0
    for op, run in cigar:
        if op == 'M' or op == 'I':
            ilen += run
        if op == 'M' or op == 'D':
            dlen += run
        totlen += run
    return ilen, totlen, dlen

# Prints span of I+M ops, then total span (I+D+M), then D+M
sizes_from_cigar(als['CM000676.2'][0].cigar)

(322166, 322198, 318507)

In [22]:
# Reference span according to GFF record
als['CM000676.2'][0].en - als['CM000676.2'][0].st + 1

318508

The length of the reference substring involved in the alignment equals the span of the `M` and `D` CIGAR operations.  We can conclude that `D` corresponds to a column where a reference character appears opposite a gap.

### Get ALT sequences

In [23]:
def parse_fasta(fh):
    fa = {}
    current_short_name = None
    # Part 1: compile list of lines per sequence
    for ln in fh:
        try:
            ln = ln.decode()
        except AttributeError:
            pass
        if ln[0] == '>':
            # new name line; remember current sequence's short name
            long_name = ln[1:].rstrip()
            current_short_name = long_name.split()[0]
            fa[current_short_name] = []
        else:
            # append nucleotides to current sequence
            fa[current_short_name].append(ln.rstrip())
    # Part 2: join lists into strings
    for short_name, nuc_list in fa.items():
        # join this sequence's lines into one long string
        fa[short_name] = ''.join(nuc_list)
    return fa

In [24]:
ftp = nav_to_root()
fastas = []
for alt_grp in alt_grps:
    dr_alt = os.path.join(alt_grp, 'alt_scaffolds', 'FASTA')
    ftp.cwd(dr_alt)
    tmp = tempfile.mkdtemp()
    fns = []

    def handle_fasta_line(st):
        fn = st.split()[-1]
        if fn == 'alt.scaf.fna.gz':
            fns.append(fn)  # pretty simple
    
    def download_fasta(fn):
        tmpfn = os.path.join(tmp, fn)
        with open(tmpfn, 'wb') as tmpfh:
            logging.info('Retrieving "%s" for group "%s"...' % (fn, alt_grp))
            ftp.retrbinary('RETR ' + fn, tmpfh.write, 1024)
            logging.info('...Done')
        assert os.path.exists(tmpfn)
        with gzip.open(tmpfn, 'rt') as fh:
            return alt_grp, parse_fasta(fh)

    ftp.retrlines('LIST', handle_fasta_line)
    for fn in fns:
        fastas.append(download_fasta(fn))
    ftp.cwd('../../..')

combined = {}
tot_len = 0
for alt_grp, fasta in fastas:
    print('%s: %d' % (alt_grp, len(fasta)))
    tot_len += len(fasta)
    combined.update(fasta)

print('Combined len = %d' % len(combined))
print('Tot of each individual = %d' % tot_len)

INFO:root:Retrieving "alt.scaf.fna.gz" for group "ALT_REF_LOCI_1"...
INFO:root:...Done
INFO:root:Retrieving "alt.scaf.fna.gz" for group "ALT_REF_LOCI_10"...
INFO:root:...Done
INFO:root:Retrieving "alt.scaf.fna.gz" for group "ALT_REF_LOCI_11"...
INFO:root:...Done
INFO:root:Retrieving "alt.scaf.fna.gz" for group "ALT_REF_LOCI_12"...
INFO:root:...Done
INFO:root:Retrieving "alt.scaf.fna.gz" for group "ALT_REF_LOCI_13"...
INFO:root:...Done
INFO:root:Retrieving "alt.scaf.fna.gz" for group "ALT_REF_LOCI_14"...
INFO:root:...Done
INFO:root:Retrieving "alt.scaf.fna.gz" for group "ALT_REF_LOCI_15"...
INFO:root:...Done
INFO:root:Retrieving "alt.scaf.fna.gz" for group "ALT_REF_LOCI_16"...
INFO:root:...Done
INFO:root:Retrieving "alt.scaf.fna.gz" for group "ALT_REF_LOCI_17"...
INFO:root:...Done
INFO:root:Retrieving "alt.scaf.fna.gz" for group "ALT_REF_LOCI_18"...
INFO:root:...Done
INFO:root:Retrieving "alt.scaf.fna.gz" for group "ALT_REF_LOCI_19"...
INFO:root:...Done
INFO:root:Retrieving "alt.scaf.fn

ALT_REF_LOCI_1: 187
ALT_REF_LOCI_10: 1
ALT_REF_LOCI_11: 1
ALT_REF_LOCI_12: 1
ALT_REF_LOCI_13: 1
ALT_REF_LOCI_14: 1
ALT_REF_LOCI_15: 1
ALT_REF_LOCI_16: 1
ALT_REF_LOCI_17: 1
ALT_REF_LOCI_18: 1
ALT_REF_LOCI_19: 1
ALT_REF_LOCI_2: 26
ALT_REF_LOCI_20: 1
ALT_REF_LOCI_21: 1
ALT_REF_LOCI_22: 1
ALT_REF_LOCI_23: 1
ALT_REF_LOCI_24: 1
ALT_REF_LOCI_25: 1
ALT_REF_LOCI_26: 1
ALT_REF_LOCI_27: 1
ALT_REF_LOCI_28: 1
ALT_REF_LOCI_29: 1
ALT_REF_LOCI_3: 7
ALT_REF_LOCI_30: 1
ALT_REF_LOCI_31: 1
ALT_REF_LOCI_32: 1
ALT_REF_LOCI_33: 1
ALT_REF_LOCI_34: 1
ALT_REF_LOCI_35: 1
ALT_REF_LOCI_4: 3
ALT_REF_LOCI_5: 3
ALT_REF_LOCI_6: 3
ALT_REF_LOCI_7: 3
ALT_REF_LOCI_8: 2
ALT_REF_LOCI_9: 1
Combined len = 261
Tot of each individual = 261


### Get a FASTA sequence

In [25]:
ident = dst_counter.most_common(1)[0][0] # CM000681.2

def fasta_from_entrez(acc):
    url = 'https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi?db=nuccore&amp;id=%s&amp;rettype=fasta&amp;retmode=text' % acc
    with urllib.request.urlopen(url) as response:
        return parse_fasta(response)

fasta = fasta_from_entrez(ident)
for k, v in fasta.items():
    print('%s, (%d): %s' % (k, len(v), v[:100]))

CM000681.2, (58617616): NNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNN


### Create a stacked alignment

In [26]:
chr19seq = fasta['CM000681.2']

In [27]:
aln = als['CM000681.2'][0]
aln

CM000681.2[20663140:21042381]->GL383573.1

In [28]:
altseq = combined['GL383573.1']
altseq[:1000]

'GTTTACCCTTAAAGGTATTGTGTGCGTGTCTTTTCTTCTCCCCTCATGTGTTTCCTGCACAGAACATATTTCTCAAAGGCTTCATTTGTTCCTTTTTATTCATTTTTGTTTAATCTTGTCTGCATGCCTTATTTTGGCAAGGTGGTCTTCAAACTCTGATGTTCTTTCTTCTGCTTGGTCAATGCAGCTACAAATACTTTTGTATGCTTCAGGAGGTTCTCATGCTGTGTGTTTTAGCTCCATCAGATCATTTATGTTCCTGTCTAAACCAGTTATTCTAGTTAGCAGTTCCTCTAACCTTTTATCAAGGTTCTTAGCTTCTTTCAATTGGTTTAGAACATGTTCCTTTAGCTCAGCAGAGTTTATTACCCATCTTCTGAAGCCTACTTTTGTCAATTCATCCATCTCATCCTCTGTCCAGTTCTGCACCCTTGCTGAAGAGACATTTTGATAACTTGGAGGAAAAGCCGCACTCTGGCCTTTCAGAGTTCTTTTGTTGATTCTTTCTCATCTTCATGAACTTGTCTAGTTTTGATCTTTGAGGCTGTTGACAACTGAATGGTTGTTTCTGTTTTGTTTTGTTTTGTTTTGAGACAGAGTCTCACTCTGTCACCCAGGCTGGAGGGCAGTGGCACAATCTCGGCTCAATGCAACCTTCGCATCCCAGGTTCAAGTGATTCTCGTGTCTCAGATTCCCGAGTAGCTGGGATTACAGGCATGCAACACAACGCCTGGCTAATTTTTGTATTTTTTAGTAGAGATGGGGTTTTGCCATGTTGGCCAGGCTAGTTTTGAACTCCTGACCTCAAGTGATCTGCCTGTCTCAGCCTCCCAAAGTACTGAGATTACTGGTGTGAGCCACCATGCCCGGCCTTGGATGGGGTTTTTGTCGGTACTTTTTTGTTGTTGATGTTGTTGTTTTCTGTTTGTTTGTTTTTCTTTCAGTGGTCAGGTCCCTCTTCTGTAGGGCTGCTGCAGTTTGCTGGGGGTTCACTTCAG

In [29]:
st, en = als['CM000681.2'][0].st, als['CM000681.2'][0].en
st, en

(20663140, 21042381)

In [30]:
# offsets were already adjusted to be 0 based when we
# constructed the Alignment object
chr19seq[st:st+30]

'GTTTACCCTTAAAGGTATTGTGTGCGTGTC'

In [31]:
altseq[:30]

'GTTTACCCTTAAAGGTATTGTGTGCGTGTC'

In [32]:
# do they seem to line up at the beginning?
chr19seq[st:st+30] == altseq[:30]
# ...yes

True

In [33]:
# Create stacked alignment out of sequences and CIGAR
def cigar_to_stacked(refseq, al):
    altid = al.target
    altseq = combined[altid]
    st, en = al.st, al.en
    refs, alts = [], []
    refsubstr = refseq[st:en]
    refoff, altoff = 0, 0
    last_op = None
    for op, run in al.cigar:
        assert last_op is None or last_op != op
        if op == 'M':
            refs.append(refsubstr[refoff:refoff+run])
            alts.append(altseq[altoff:altoff+run])
            refoff += run
            altoff += run
        elif op == 'D':
            refs.append(refsubstr[refoff:refoff+run])
            alts.append('-' * run)
            refoff += run
        elif op == 'I':
            refs.append('-' * run)
            alts.append(altseq[altoff:altoff+run])
            altoff += run
        last_op = op
    return ''.join(refs), ''.join(alts)

In [34]:
# For prettier printing of these large stacked alignments
def flow_stacked(refs, alts, per_line=80):
    off = 0
    lines = []
    while off < len(refs):
        refline = refs[off:off+per_line]
        altline = alts[off:off+per_line]
        bars = []
        for rc, ac in zip(refline, altline):
            bars.append('|' if rc == ac else ' ')
        lines.append(refline)
        lines.append(''.join(bars))
        lines.append(altline)
        lines.append('\n')
        off += per_line
    return lines

In [35]:
# Try for first ALT locus on chr19
lines = flow_stacked(*cigar_to_stacked(chr19seq, als['CM000681.2'][0]))
print('\n'.join(lines[7600:7700]))

TTTTTTTTTCAGAAACAACAGCTTAGAAAACACCAGAGAGTTTATACTAAAGTATAATTTTGCAGATGCAGTAAATATAA
||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||
TTTTTTTTTCAGAAACAACAGCTTAGAAAACACCAGAGAGTTTATACTAAAGTATAATTTTGCAGATGCAGTAAATATAA


AAACAATTCAAAATAGAATGTATATAAATGTCAGAATTTACAGTAGAATAACTAAGGCACTGACACTTAAGACATTACAC
||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||
AAACAATTCAAAATAGAATGTATATAAATGTCAGAATTTACAGTAGAATAACTAAGGCACTGACACTTAAGACATTACAC


TAAATCAGAGTGTTGAGTATAAAAACTAATCCACAACTACAGTTTTTAGATAAATGATTTGTATGTAACTTTAAAA--AG
|||||||||||||||| |||||||||||||||||||||||||||||||||||||||||||||||||||||||||||  ||
TAAATCAGAGTGTTGACTATAAAAACTAATCCACAACTACAGTTTTTAGATAAATGATTTGTATGTAACTTTAAAAAGAG


ATTTTTGGAAGCATTGTAATTACATTGAAAGTACACTTGTTTCCTTGAATAAAATTTTTTGAAAACTGAATAATGATATA
|||||||||||||||||| |||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||
ATTTTTGGAAGCATTGTAGTTACATTGAAAGTACACTTGTTTCCTTGAATAAAATTTTTTGAAAACTGAATAATGATATA


ATACAGCTTTCAAATTACTT

In [36]:
# Try for second ALT locus on chr19
lines = flow_stacked(*cigar_to_stacked(chr19seq, als['CM000681.2'][1]))
print('\n'.join(lines[3700:3800]))

TGTTTCCTCAGATGGGGTTCAAAGACAACAAGGTGTCTGAATGACTCAGCATAATGCTCTCTCTCCTCTCCCCCTTTCAT
||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||
TGTTTCCTCAGATGGGGTTCAAAGACAACAAGGTGTCTGAATGACTCAGCATAATGCTCTCTCTCCTCTCCCCCTTTCAT


TTTCTCCTTTCTCCCTCTCCTGTCCCCATCTCTTTCTC-TTTTTTTTTTTTTTTTTTTTTTTACCTTTCAGAAAACCACA
|||||||||||||||||||||||||||||||||||||| |||||||||||||||||||||||||||||||||||||||||
TTTCTCCTTTCTCCCTCTCCTGTCCCCATCTCTTTCTCTTTTTTTTTTTTTTTTTTTTTTTTACCTTTCAGAAAACCACA


GCTTTTGGACCCTAAAAGGTCTGGATTGATCGTACTGCTTTCTGAAAGGTAAAAAAAACAAATACTTTGGGAGGTGTGAA
||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||
GCTTTTGGACCCTAAAAGGTCTGGATTGATCGTACTGCTTTCTGAAAGGTAAAAAAAACAAATACTTTGGGAGGTGTGAA


GGTTGGCTTTGTGGGGGACAGCAGGTTCATCTTGTAGCTCATTGCTGTCAACTTATGCCATACCATATGTATTTCAGTTT
||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||
GGTTGGCTTTGTGGGGGACAGCAGGTTCATCTTGTAGCTCATTGCTGTCAACTTATGCCATACCATATGTATTTCAGTTT


AATAACGGGGTGCAAATGTT